# The PFT and IPFT pipeline

`pypft.forward_pft`/`pypft.inverse_pft` compose the pieces from every earlier
notebook -- the angular DFT and the discrete Hankel transform (DHT) -- into
the full chain from the front page of `README.md`:

$$f(r, \theta) \xrightarrow{\text{DFT}_\theta} f_n(r) \xrightarrow{H_n} F_n(\rho) \xrightarrow{\text{IDFT}_\varphi} F(\rho, \varphi)$$

This notebook runs that chain against a known analytic pair -- a radially
symmetric Gaussian and its own continuous Fourier transform -- and
reproduces the accuracy figures Yao & Baddour publish for it
[YaoBaddour2020, Appendix A-5/A-6].


In [ ]:
import matplotlib.pyplot as plt
import numpy as np

import pypft


## Building the grid and a test signal

`forward_pft`/`inverse_pft` take a `pypft.PolarGrid` and a `(n_radial,
n_angular)` array on PyPFT's own axis layout (`pypft.Axis`) -- the *opposite*
of `grid.r`'s own `(n_angular, n_radial)` layout from the previous notebook
(each row of `grid.r` is one harmonic's own radial samples, the natural
shape for *building* the grid, not for storing a signal). `grid.r.T` gives
the transpose PyPFT's own layout expects.

The parameters below -- `n_radial=382` (Baddour's `N1=383`), `n_angular=15`,
`R=40` -- are Yao & Baddour's own worked example [YaoBaddour2020, Appendix
A-5].

In [ ]:
grid = pypft.PolarGrid(n_radial=382, n_angular=15, R=40.0)

f = np.exp(-(grid.r.T**2))  # a radially symmetric Gaussian, f(r) = exp(-r^2)
F = pypft.forward_pft(f, grid)
f.shape, F.shape


## Comparing to the analytic oracle

A radially symmetric Gaussian is one of the few functions with a
closed-form continuous 2-D Fourier transform: $f(r) = e^{-r^2}$ transforms
to $F(\rho) = \pi e^{-\rho^2/4}$ [YaoBaddour2020, Appendix A-5]. Comparing
the discrete `F` above to this analytic answer, in dB, is exactly how the
paper itself reports accuracy:

In [ ]:
expected = np.pi * np.exp(-(grid.rho.T**2) / 4.0)
err_db = 20 * np.log10(np.abs(expected - F) / np.max(np.abs(F)))

print(f"average error: {err_db.mean():.2f} dB")
print(f"maximum error: {err_db.max():.2f} dB")


These match Yao & Baddour's own published figures for this exact
`(n_angular, n_radial, R)` combination almost to the decimal
[YaoBaddour2020, Appendix A-5]. Note the *maximum* error is much worse than
the average: it occurs at the grid's central gap (see the previous
notebook), where the sampling grid is unavoidably sparse. **Every accuracy
claim about the PFT should be about the average error, never the max.**

In [ ]:
theta_grid = np.broadcast_to(grid.theta[np.newaxis, :], grid.rho.T.shape)

fig, ax = plt.subplots(figsize=(5, 5), subplot_kw={"projection": "polar"})
scatter = ax.scatter(theta_grid.ravel(), grid.rho.T.ravel(), c=err_db.ravel(), s=6)
ax.set_title("Forward PFT error (dB)")
fig.colorbar(scatter, ax=ax, label="dB")
plt.show()


## The inverse transform

`inverse_pft` retraces the same three steps in the opposite direction:
$F(\rho, \varphi) \xrightarrow{\text{DFT}_\varphi} F_n(\rho)
\xrightarrow{H_n} f_n(r) \xrightarrow{\text{IDFT}_\theta} f(r, \theta)$.
Feeding the *analytic* frequency-domain oracle in and comparing against the
analytic space-domain one is the same kind of check as above, this time in
the other direction [YaoBaddour2020, Appendix A-6]:

In [ ]:
F_oracle = np.pi * np.exp(-(grid.rho.T**2) / 4.0)
f_reconstructed = pypft.inverse_pft(F_oracle, grid)

f_oracle = np.exp(-(grid.r.T**2))
inverse_err_db = 20 * np.log10(
    np.abs(f_oracle - f_reconstructed) / np.max(np.abs(f_reconstructed))
)

print(f"average error: {inverse_err_db.mean():.2f} dB")
print(f"maximum error: {inverse_err_db.max():.2f} dB")


## Round trip is a regression check, not an accuracy check

`inverse_pft(forward_pft(f, grid), grid)` returns to `f` almost exactly,
*regardless of how accurate the forward transform actually was* -- the DHT's
self-inverse kernel and the forward/inverse scale factors cancel exactly
across a round trip [YaoBaddour2020, Appendix A-5/A-6]. A passing round trip
is therefore useful only for catching a regression in the composition
itself; it can never certify that the forward or inverse transform is
numerically accurate (the two checks above are what do that).

In [ ]:
round_tripped = pypft.inverse_pft(pypft.forward_pft(f, grid), grid)
float(np.abs(round_tripped - f).max())


## Where to go next

This is the full PFT/IPFT pipeline this package builds towards: an angular
DFT, a per-harmonic discrete Hankel transform, and an angular IDFT, composed
on top of `PolarGrid`. Later notebooks will cover the transform's
analytical properties, typed domains, batching, and visualization as they
land.

In [ ]:
from IPython.display import Markdown, display

from pypft.references import Reference, bibliography

display(Markdown(bibliography(Reference.YAO_BADDOUR_2020_PFT_PART2)))
